# Basketball MARKET coverage: separate season, composition and support

This notebook reviews **sanitized saved-prediction aggregates** from 2025–2026. No real fit, model inference, new source request or private game row is used. B005 completed zero-fit evidence recovery; B003 remains the latest fully completed real modeling release. B004 remains stopped, and 238 cumulative real units are preserved.

The NCAA men comparison originally pooled uncovered 2025–2026 games against a covered panel entirely from 2026. We first compare the same season. Baseline-probability matching alone leaves the gap; month-aware matching substantially reduces it **and removes uncovered support**. This is exploratory description, not a causal effect of coverage, equivalence test, model promotion or MARKET advantage.

[Full report](../docs/BASKETBALL_MARKET_COVERAGE.md) · [Aggregate evidence](../docs/evidence/2026-10-28-basketball-market-coverage.json)

In [1]:
from pathlib import Path
import json
from html import escape
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import display, HTML
from examples.basketball_contracts.evidence_market_coverage import validate
pio.renderers.default = "plotly_mimetype"
root = Path.cwd()
if not (root / "docs/evidence").exists():
    root = root.parent
data = json.loads((root / "docs/evidence/2026-10-28-basketball-market-coverage.json").read_text())
receipt = validate(data)
print(json.dumps(receipt, sort_keys=True))
assert data["historical_years"] == [2025, 2026] and not data["current_2027_used"]
print("Saved aggregate arithmetic verified; 2027 outcomes excluded; no new fits or source requests.")

{"new_real_fits": 0, "standardizations": 15, "supported_strata": 194, "year_coverage_rows": 17}
Saved aggregate arithmetic verified; 2027 outcomes excluded; no new fits or source requests.


## Same-season controlled comparisons

Negative log-loss deltas favor the rich candidate over the league control: selected NBA strength, and exact B002 combined in both colleges. NBA rich remains a rejected candidate; no champion changes. Covered means a valid saved T60 sportsbook benchmark. Uncovered does not establish that no real-world sportsbook quote existed.

The intervals are ordinary seven-day calendar-block bootstrap intervals, retained from the private saved-output audit. Aggregate arithmetic is reproducible publicly; the original block resamples require the private row data. NBA's 9 and 16 covered games are especially weak support.

In [2]:
names = {"nba":"NBA", "ncaam":"NCAA men", "ncaaw":"NCAA women"}
years = [r for r in data["year_coverage"] if r["year"] != "pooled"]
fig = go.Figure()
for covered, color in ((0,"#4063b0"),(1,"#c45732")):
    rows = [r for r in years if r["covered"] == covered]
    x = [r["delta_mean"] for r in rows]
    fig.add_trace(go.Scatter(x=x, y=[f'{names[r["league"]]} {r["year"]}' for r in rows],
        name="Covered" if covered else "Uncovered", mode="markers", marker={"color":color,"size":10},
        error_x={"type":"data","symmetric":False,
            "array":[r["delta_ci"]["ci95"][1]-r["delta_mean"] for r in rows],
            "arrayminus":[r["delta_mean"]-r["delta_ci"]["ci95"][0] for r in rows]},
        customdata=[[r["base_n"],r["base_log_loss"],r["rich_log_loss"]] for r in rows],
        hovertemplate="%{y}<br>Delta=%{x:.9f}<br>Games=%{customdata[0]}<br>Control=%{customdata[1]:.9f}<br>Rich=%{customdata[2]:.9f}<extra>%{fullData.name}</extra>"))
fig.add_vline(x=0,line_color="#555",line_dash="dot")
fig.update_layout(title="Rich − league-control log loss, exact cohorts by year",xaxis_title="Negative favors rich combined",height=470,template="plotly_white",legend={"orientation":"h"},margin={"l":125,"r":20,"t":70,"b":60})
fig.show()

In [3]:
def html_table(headers, rows):
    head="".join("<th>"+escape(str(x))+"</th>" for x in headers)
    body="".join("<tr>"+"".join("<td>"+escape(str(x))+"</td>" for x in row)+"</tr>" for row in rows)
    return HTML("<table><thead><tr>"+head+"</tr></thead><tbody>"+body+"</tbody></table>")
display(html_table(["League","Year","Covered","Games","Control LL","Rich LL","Delta"],
    [[names[r["league"]],r["year"],bool(r["covered"]),r["base_n"],f'{r["base_log_loss"]:.9f}',f'{r["rich_log_loss"]:.9f}',f'{r["delta_mean"]:+.9f}'] for r in years]))
print("NCAA men has no 2025 covered comparison. Same-2026 raw covered-minus-uncovered delta:")
print(json.dumps(data["ncaam_2026_raw_gap"],indent=2))

NCAA men has no 2025 covered comparison. Same-2026 raw covered-minus-uncovered delta:
{
  "covered_minus_uncovered_delta": 0.008515371409992102,
  "ci95": [
    0.0029931663749820864,
    0.014439907382806653
  ],
  "valid_repetitions": 2000,
  "blocks": 23,
  "method": "shared_7_day_calendar_block_bootstrap",
  "seed": 6106
}


League,Year,Covered,Games,Control LL,Rich LL,Delta
NBA,2025,False,1239,0.612608757,0.614320226,+0.001711470
NBA,2025,True,9,0.603386008,0.598678597,-0.004707410
NBA,2026,False,1233,0.601841739,0.601299031,-0.000542709
NBA,2026,True,16,0.645275550,0.654399998,+0.009124447
NCAA men,2025,False,5758,0.530652365,0.524171133,-0.006481232
NCAA men,2026,False,4606,0.522552995,0.514027624,-0.008525371
NCAA men,2026,True,1143,0.577580458,0.577570458,-0.000010000
NCAA women,2025,False,5131,0.463780170,0.456514714,-0.007265456
NCAA women,2025,True,149,0.433657861,0.430101008,-0.003556853
NCAA women,2026,False,5478,0.466506477,0.457584957,-0.008921520


## Calendar composition and common-support loss

The covered men panel is not uniformly “more competitive.” Its mean baseline favorite confidence is lower (0.7172 versus 0.7326), but its central 0.4–0.6 probability share is **22.13% versus 23.01%**. Probability-quintile reweighting leaves a +0.008104 gap.

January–February accounts for **87.66% covered versus 41.12% uncovered**. November has only 1 covered game versus 1,241 uncovered games. Month-aware comparisons have much weaker overlap.

The chart compares the full raw gap with within-stratum gaps. These are **different estimands when support changes**. Error bars are exploratory, unadjusted intervals from development-exposed seasons. Raw intervals resample shared calendar blocks; adjusted intervals use positive exponential shared-block multipliers with fixed support. An interval crossing zero does not establish equivalence. This independent forensic analysis uses minimum five plus minimum-twenty sensitivities. The offline B006 owner default uses minimum twenty with its own declared confidence strata, a separate specification.

In [4]:
lookup = {s["spec"]:s for s in data["standardizations"]}
chosen = [("Probability quintile","probability_quintile"),("Month","month"),("Probability × month × game type","probability_month_game_type"),("Same, minimum 20 per cohort","probability_month_game_type_min20")]
raw=data["ncaam_2026_raw_gap"]
labels=["Raw full 2026 cohort"]+[label for label,_ in chosen]
values=[raw["covered_minus_uncovered_delta"]]+[lookup[key]["within_stratum_component_covered_reference"] for _,key in chosen]
intervals=[raw["ci95"]]+[lookup[key]["descriptive_intervals"]["within_stratum_component"] for _,key in chosen]
retained=[[100.,100.,4606.]]+[[100*lookup[key]["covered_retention"],100*lookup[key]["uncovered_retention"],lookup[key]["uncovered_reweighting_ess"]] for _,key in chosen]
fig=go.Figure(go.Scatter(x=values,y=labels,mode="markers",marker={"size":11,"color":["#555"]+["#197b6b"]*4},
    error_x={"type":"data","symmetric":False,"array":[ci[1]-v for ci,v in zip(intervals,values)],"arrayminus":[v-ci[0] for ci,v in zip(intervals,values)]},
    customdata=retained,hovertemplate="%{y}<br>Gap=%{x:.9f}<br>Covered retained=%{customdata[0]:.2f}%<br>Uncovered retained=%{customdata[1]:.2f}%<br>Uncovered ESS=%{customdata[2]:.1f}<extra></extra>"))
fig.add_vline(x=0,line_dash="dot",line_color="#555")
fig.update_layout(title="NCAA men 2026: within-stratum gap and changing support",xaxis_title="Covered − uncovered rich-model improvement (positive means less covered benefit)",height=440,template="plotly_white",margin={"l":265,"r":30,"t":65,"b":80},showlegend=False)
fig.show()
print("Primary month-aware retained counts: 1,137 covered / 3,266 uncovered; ESS 2,289.0.")
print("Raw 0.008515371 = support exclusion 0.003423843 + composition 0.003262061 + within 0.001829468.")

Primary month-aware retained counts: 1,137 covered / 3,266 uncovered; ESS 2,289.0.
Raw 0.008515371 = support exclusion 0.003423843 + composition 0.003262061 + within 0.001829468.


In [5]:
display(html_table(["Declared grouping","Covered retained","Uncovered retained","Uncovered ESS","Within gap","Exploratory interval"],
    [[s["spec"],f'{s["covered_retention"]:.2%}',f'{s["uncovered_retention"]:.2%}',f'{s["uncovered_reweighting_ess"]:.1f}',f'{s["within_stratum_component_covered_reference"]:+.9f}',"["+", ".join(f"{v:+.9f}" for v in s["descriptive_intervals"]["within_stratum_component"])+"]"] for s in data["standardizations"]]))
print("All 15 declared descriptive sensitivity specifications are shown; no result selects a model.")

All 15 declared descriptive sensitivity specifications are shown; no result selects a model.


Declared grouping,Covered retained,Uncovered retained,Uncovered ESS,Within gap,Exploratory interval
probability_quintile,100.00%,100.00%,4472.6,+0.008104461,"[+0.003198279, +0.013848060]"
probability_decile,100.00%,100.00%,4329.7,+0.008414666,"[+0.003522621, +0.014367593]"
favorite_confidence,100.00%,100.00%,4329.7,+0.008414666,"[+0.003522621, +0.014367593]"
month,99.91%,72.77%,2363.4,+0.001553897,"[-0.001923927, +0.004647551]"
game_type,100.00%,100.00%,4522.9,+0.008542960,"[+0.003233058, +0.014720643]"
neutral,100.00%,100.00%,4140.6,+0.008768922,"[+0.003333937, +0.014991952]"
probability_month_game_type,99.48%,70.91%,2289.0,+0.001829468,"[-0.002102329, +0.004976417]"
confidence_favorite_month_game_type,96.59%,55.73%,2164.7,+0.003012435,"[-0.000515032, +0.006018381]"
probability_month_game_type_neutral,98.78%,67.98%,2234.4,+0.001997879,"[-0.001378546, +0.005187397]"
probability_decile_month_game_type,96.59%,55.73%,2164.7,+0.003012435,"[-0.000515032, +0.006018381]"


## Error diagnosis and boundaries

On covered men games, rich combined fixes 22 baseline incorrect picks but loses 23 correct picks; on uncovered games the counts are 131 and 113. Updates move toward the realized winner on 45.49% versus 55.67% of games. These are outcome-conditioned diagnostics, not pregame signals.

Rich combined is closer to sportsbook probabilities (mean absolute disagreement 0.04258 versus 0.05253), yet remains 0.010443758 worse in log loss. Its probability-error correlation with MARKET is 0.99317; shared subtraction of the same outcome mechanically raises this correlation. No independent MARKET information, fitted residual edge, historical exchange trading backtest, order, fill or profit is established.

Feature-availability age is league-state recency under reconstructed 24-hour research timing, not per-team feature freshness or original publication time. Frozen feature matrices were not exported; per-game completeness cannot be attributed from these predictions. Unverified conference tags are excluded from public matching.

The measured active floor is **4,250.265382922 seconds**. The open finalization reservation extends to **4,430.265382922**, which is not measured spent time. Conservative remaining work is 69.734617078 seconds; source requests are 48/48. Future source or training work requires explicit budget approval. The 300-base/80-calibration requirements and stability gates remain in force. All 330 prior forecast versions remain preserved. 2027 outcomes cannot choose a model. The rating remains 8.5/10.

In [6]:
# A later cell proves that both figures and exact tables did not interrupt execution.
assert validate(data)["standardizations"] == 15
owner=data["owner_recovery"]
assert owner["new_real_fits"] == 0 and owner["cumulative_real_basketball_units"] == 238
assert not data["next_research"]["extra_training_authorized"]
print("MARKET_COVERAGE_REVIEW_COMPLETE")
print("Aggregate validation complete; no promotion, new forecast, private rows or additional budget.")

MARKET_COVERAGE_REVIEW_COMPLETE
Aggregate validation complete; no promotion, new forecast, private rows or additional budget.
